In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Launching SWAN Cases with `swanpy`

## Imports

In [2]:
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

from oceanicospy.models.swanpy import Initializer
from oceanicospy.models.swanpy.preprocess import GridMaker, BathyMaker

## Stationary case:

This notebook walks through a complete SWAN **stationary** modelling workflow using the `swanpy` subpackage inside `oceanicospy`. The example is based on Hurricane Matthew (October 2016) over the Guajira region.

**Workflow overview**

1. [Imports & case parameters](#1-imports--case-parameters)
2. [Initial setup](#2-initial-setup)
3. [Grid](#3-grid)
4. [Bathymetry](#4-bathymetry)
5. [Wind forcing](#5-wind-forcing)
6. [Boundary conditions](#6-boundary-conditions)
7. [Physics](#7-physics)
8. [Numerics](#8-numerics)
9. [Computation & run configuration](#9-computation--run-configuration)

```{note}
The workflow assumes that input data (bathymetry `.dat`, ERA5 NetCDF files) are already placed in the appropriate `input/domain_01/` folder inside the case directory.

### Case parameters definition

Three main parameters are required to set up a SWAN case:

| Parameter | Description |
|---|---|
| `path_case` | Root directory that will hold `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` / `end_date` | Simulation window |
| `peak_date` | Date of maximum forcing (stationary snapshot) |

The initialization metadata dictionary (`dict_ini_data`) is a key component of the workflow, as it contains all the necessary information to retrieve and process input data (e.g., ERA5 files). This dictionary is expetected to have the following keys:

- name: A string identifier for the case (e.g., "hurricane_matthew_guajira_2016").
- case_number (optional): An integer to differentiate between multiple cases (e.g., 1, 2, 3).
- case_description: A brief description of the case (e.g., "Stationary", "Non-stationary").
- stat_id: An integer indicating the type of run (0 for non-stationary, 1 for stationary).
- number_domains: An integer specifying the number of domains in the simulation (e.g., 1, 2, 3). 
- nested_domains: number of nested domains (e.g., 0, 1, 2) if applicable.
- parent_domains: a dictionary specifying the parent domain for each nested domain (e.g., {"domain_02": "domain_01", "domain_03": "domain_02"}). If there are no nested domains, this can be set to None or an empty dictionary.

 The expected structure of this dictionary is detailed in the `Initializer` documentation.

In [3]:
# ── User settings ─────────────────────────────────────────────────────────────
path_case = '../data/model_runs_data/matthew2016/'

dict_ini_data = dict(
    name            = 'Termoguajira',
    case_number     = 1,
    case_description= 'Matthew 2016',
    stat_id         = 1,
    number_domains  = 1,
    nested_domains  = 0,
    parent_domains  = {1: None},
)

ini_date  = dt.datetime(2016,  9, 27)
peak_date = dt.datetime(2016, 10,  1)   # stationary snapshot date
end_date  = dt.datetime(2016, 10,  4)

# ──────────────────────────────────────────────────────────────────────────────

```{hint}
`stat_id` convention
* `stat_id = 0` → **non-stationary** run  
* `stat_id = 1` → **stationary** run  ← _used in this notebook_

### Initial setup

`Initializer` class creates the project directory tree and writes the baseline SWAN configuration file (`run.swn`) from a template. The tree has a structure like the following, where `domain_01` is the main domain (and only domain in this case):

```
path_case/
├── input/
│   └── domain_01/          ← place ERA5 .nc and bathymetry .dat files here
├── pros/
├── run/
│   └── domain_01/
│       └── run.swn         ← generated from template
└── output/
    └── domain_01/
```



All of this code is expected to be in a python file inside the `pros/` folder so `pros/` is the working directory when running the script and it should be already created before calling `Initializer`. The folder `input/domain_0X/` should also be created and populated with the necessary input files (ERA5 NetCDF files and bathymetry `.dat` file) before running the script.

```{note}

```{note}
Calling `create_folders_l2()` is only required when running **nested** domains.  
For a single-domain run `create_folders_l1()` is sufficient.

In [4]:
case = Initializer(
    root_path    = path_case,
    dict_ini_data= dict_ini_data,
    ini_date     = ini_date,
    end_date     = end_date,
)

case.create_folders()    # creates all project folders (data/, output/, etc.)
case.replace_ini_data()     # writes run.swn from the STAT template

*** Initializing SWAN model ***


	*** Creating project folder structure ***


	*** Copying base SWAN configuration file into run folder ***



### Creating the grid

`GridMaker` is a class inside `swanpy` that populates the `CGRID` and `INPGRID` sections of configuration file `run.swn`. 

Two methods are available to obtain the grid parameters:

| Method | When to use |
|---|---|
| `params_from_bathy()` | Derive extent and resolution automatically from the `.dat` bathymetry file |
| `params_from_user()` | Supply the grid dictionary yourself (full control) |

```{note}
If the params_from_bathy() method is used, the grid parameters will be automatically derived from the `.dat` bathymetry file. In this case, the user must ensure that the bathymetry file is correctly formatted and placed in the appropriate `input/domain_0X/` folder before running the script.

When the user provides the grid parameters, the following keys are expected in the `grid_info` dictionary:
- `lon_ll_corner`: longitude of the lower-left corner of the grid
- `lat_ll_corner`: latitude of the lower-left corner of the grid
- `x_extent`: total extent of the grid in the x-direction (longitude)
- `y_extent`: total extent of the grid in the y-direction (latitude)
- `nx` : number of grid cells in the x-direction
- `ny`: number of grid cells in the y-direction


In [5]:
DOMAIN_ID = 1   # domain identifier (1-indexed)

case_grid = GridMaker(
    init         = case,
    domain_number= DOMAIN_ID,
    dx           = 100,    # grid spacing in x [m]
    dy           = 100,    # grid spacing in y [m]
)

# # Option A – derive grid parameters from the bathymetry file
# dict_grid = case_grid.params_from_bathy()

# # Option B – supply your own grid dictionary (uncomment to use)
dict_grid = case_grid.params_from_user()   # requires grid_info= at construction

# print(dict_grid)
# case_grid.fill_grid_section(dict_grid)


*** Initializing gridmaker for domain 1 ***



AttributeError: 'GridMaker' object has no attribute 'params_from_user'

### Setting up the bathymetry

`BathyMaker` class fills the `BOTTOM` section of `run.swn`. There are two methods to set up the bathymetry file for the model:

* `get_direct_from_user` is used when a pre-processed `.bot` file already exists in `input/domain_01/`. This method implies the use of a dictionary with bathymetry metadata.
* `xyz2asc` reads a three-column (`lon lat depth`) `.dat` file, interpolates it onto the model grid, and writes an ESRI ASCII `.bot` file to the run directory.

```{note}

```{note}
If the get_direct_from_user method is used, make sure that the pre-processed `.bot` file is correctly formatted and placed in the appropriate `input/domain_0X/` folder before running the script. This method will link or copy the existing `.bot` file to the run directory and update the bathymetry information in the configuration metadata, but it does not perform any processing on the file itself.

In [10]:
case_bathy = BathyMaker(
    init         = case,
    domain_number= DOMAIN_ID,
    filename     = 'bathymetry',   # output .bot file will be named bathymetry.bot
    bathy_info   = {'lat_ll_bat_corner': -90, 'lon_ll_bat_corner': -180, 
                    'x_bot': 100, 'y_bot': 100, 
                    'spacing_x': 100, 'spacing_y': 100},     
)

dict_bathy = case_bathy.get_direct_from_user()

print(dict_bathy)
case_bathy.fill_bathy_section(dict_bathy)


*** Initializing bathymaker for domain 1 ***

../data/model_runs_data/matthew2016/input/domain_01/*.bot
Found bathymetry files: ['../data/model_runs_data/matthew2016/input/domain_01/bathymetry.bot']
{'lat_ll_bat_corner': -90, 'lon_ll_bat_corner': -180, 'x_bot': 100, 'y_bot': 100, 'spacing_x': 100, 'spacing_y': 100, 'bathy_file': '../data/model_runs_data/matthew2016/input/domain_01/bathymetry.bot'}

 	*** Adding/Editing bathymetry information for domain 1 in configuration file ***



> **Tip:** If you already have a ready-to-use `.bot` file in `input/domain_01/`, call `case_bathy.get_from_user()` instead of `xyz2asc`. Pass the metadata dict (or `None`) as `bathy_info=` at construction time and use `use_link=True` to symlink rather than copy the file into the run directory.

## 5. Wind forcing

`WindForcing` handles ERA5 or CMDS wind data and writes the `WIND` section of `run.swn`.

**Typical workflow:**
1. Define the wind grid metadata in `wind_info`.
2. (Optional) Download ERA5 winds with `get_winds_from_ERA5()`.
3. Convert the NetCDF file to the SWAN ASCII format with `write_ERA5_ascii()`.
4. Write the section to `run.swn` with `fill_wind_section()`.

The `share_winds=True` flag means domain 1 downloads the wind file and all other domains reuse it via symlinks.

In [ ]:
# Spatial and temporal metadata for the wind grid
wind_info = dict(
    lon_ll_wind    = -74.5,   # lower-left longitude of wind grid [deg]
    lat_ll_wind    =  11.0,   # lower-left latitude  of wind grid [deg]
    meshes_x_wind  = 10,      # number of cells in x
    meshes_y_wind  =  8,      # number of cells in y
    dx_wind        = 0.25,    # grid spacing in x [deg]  (ERA5 native ~0.25°)
    dy_wind        = 0.25,    # grid spacing in y [deg]
    ini_wind_date  = ini_date.strftime('%Y%m%d.%H%M%S'),
    dt_wind_hours  = 1,
    end_wind_date  = end_date.strftime('%Y%m%d.%H%M%S'),
)

case_winds = WindForcing(
    init         = case,
    domain_number= DOMAIN,
    wind_info    = wind_info,
    share_winds  = True,
)

# Optional: download winds from ERA5 (skip if the .nc file already exists)
# difference_to_UTC = -5   # Colombia Standard Time offset
# case_winds.get_winds_from_ERA5(difference_to_UTC=difference_to_UTC, filename='winds_era5.nc')

# Convert NetCDF → SWAN ASCII wind file and update wind_info with file path
dict_winds = case_winds.write_ERA5_ascii(
    era5_filename = 'winds_era5.nc',
    ascii_filename= 'winds.wnd',
)

print(dict_winds)
case_winds.fill_wind_section(dict_winds)

## 6. Boundary conditions

`BoundaryConditions` writes the `BOUNDARY` section of `run.swn` using **TPAR** (parametric spectral) files — one per boundary point.

**Typical workflow:**
1. (Optional) Download ERA5 wave data with `get_waves_from_ERA5()`.
2. Generate `.bnd` TPAR files with `tpar_from_ERA5()` for a set of lat/lon boundary points.
3. Write the boundary section with `fill_boundaries_section()`.

The boundary points below sample the four sides of the Guajira domain.

In [ ]:
# Lat/lon points used to sample wave conditions along each boundary side
# ERA5 will be queried at (max_lat, lon), (min_lat, lon), (lat, max_lon), (lat, min_lon)
boundary_lats = [11.0, 11.5, 12.0, 12.5]   # latitudes along W/E sides
boundary_lons = [-74.5, -74.0, -73.5]       # longitudes along N/S sides

case_bounds = BoundaryConditions(
    init         = case,
    domain_number= DOMAIN,
    input_filename= 'waves_era5.nc',
    list_sides   = ['N', 'S', 'E', 'W'],
)

# Optional: download ERA5 wave data (skip if the file already exists)
# difference_to_UTC = -5
# case_bounds.get_waves_from_ERA5(difference_to_UTC, wind_info_dict=wind_info, filename='waves_era5.nc')

# Generate one .bnd TPAR file per boundary point
case_bounds.tpar_from_ERA5(
    points_lat= boundary_lats,
    points_lon= boundary_lons,
)

# Inject BOUNDARY commands into run.swn
case_bounds.fill_boundaries_section()

## 7. Physics

`PhysicsMaker` controls wave generation and dissipation settings (the `GEN` and related lines in `run.swn`).

| Parameter | Description |
|---|---|
| `cds1` | Drag coefficient for whitecapping (Janssen) |
| `delta` | SWAN tuning parameter δ (wave age) |

Leaving both empty keeps the SWAN defaults.

In [ ]:
case_physics = PhysicsMaker(init=case, domain_number=DOMAIN)

# define_generation returns a dict; pass it straight to fill_physics_section
dict_physics = case_physics.define_generation(
    cds1 ='',   # leave empty for SWAN default
    delta='',   # leave empty for SWAN default
)

case_physics.fill_physics_section(dict_physics)

## 8. Numerics

`NumericsMaker` sets the convergence / stopping criteria (`NUM` line in `run.swn`).

| `stop_criteria` | Meaning |
|---|---|
| `'STOPC'` | Stop on percentage of change (default) |
| `'ACCUR'` | Stop on absolute accuracy |

`iters` specifies the maximum number of iterations (leave empty for the SWAN default).

In [ ]:
case_numerics = NumericsMaker(init=case, domain_number=DOMAIN)

dict_numerics = case_numerics.define_stopping_criteria(
    stop_criteria='STOPC',
    iters='',           # empty → SWAN default maximum iterations
)

case_numerics.fill_numerics_section(dict_numerics)

## 9. Computation & run configuration

`CaseRunner` handles the final steps:

* `define_output_from_file()` – reads a CSV of output point coordinates and writes a `.loc` file.
* `fill_computation_section()` – writes the `COMPUTE` command (stationary or non-stationary).
* `fill_slurm_file()` – (HPC only) generates a SLURM launcher script.

### Stationary computation dictionary

For a **stationary** run set `stat_comp=1` and provide a list of `comp_dates` (one entry per snapshot). Setting `init_intermediate=True` inserts an `INIT` command between consecutive stationary computations to reinitialize the wave field.

In [ ]:
# Stationary computation: one snapshot at the peak of the event
comp_data_stat = dict(
    stat_comp        = 1,               # 1 → stationary
    comp_dates       = [peak_date],     # list of dt.datetime objects
    init_intermediate= False,           # True → insert INIT between snapshots
    # Output window (used by output definition, not strictly required for STAT)
    ini_output_date  = ini_date.strftime('%Y%m%d.%H%M%S'),
    output_res_min   = 60,
)

case_run = CaseRunner(
    init         = case,
    domain_number= DOMAIN,
    dict_comp_data= comp_data_stat,
    all_domains  = {},   # pass a domain-info dict when using nested domains
)

# (Optional) define point output locations from a CSV with X, Y columns
# case_run.define_output_from_file(filename='output_points.csv')

# (Optional) write nesting output section
# case_run.write_nest_section()

# Write the COMPUTE command and clean up any remaining template placeholders
case_run.fill_computation_section()

# (Optional) generate SLURM launcher for HPC submission
# case_run.fill_slurm_file()

---

## Summary

After running all cells the `run/domain_01/` folder contains a fully populated `run.swn` file ready for SWAN execution. The overall call sequence is:

```python
# 1. Setup
case = Initializer(root_path, dict_ini_data, ini_date, end_date)
case.create_folders_l1() ; case.create_folders_l2() ; case.replace_ini_data()

# 2. Grid
case_grid = GridMaker(case, DOMAIN, dx=100, dy=100)
case_grid.fill_grid_section(case_grid.params_from_bathy())

# 3. Bathymetry
case_bathy = BathyMaker(case, DOMAIN, filename='bathymetry', dx_bat=100)
case_bathy.fill_bathy_section(case_bathy.xyz2asc(nodata_value=-9999))

# 4. Wind
case_winds = WindForcing(case, DOMAIN, wind_info=wind_info)
case_winds.fill_wind_section(case_winds.write_ERA5_ascii('winds_era5.nc', 'winds.wnd'))

# 5. Boundary conditions
case_bounds = BoundaryConditions(case, DOMAIN)
case_bounds.tpar_from_ERA5(boundary_lats, boundary_lons)
case_bounds.fill_boundaries_section()

# 6. Physics
case_physics = PhysicsMaker(case, DOMAIN)
case_physics.fill_physics_section(case_physics.define_generation())

# 7. Numerics
case_numerics = NumericsMaker(case, DOMAIN)
case_numerics.fill_numerics_section(case_numerics.define_stopping_criteria())

# 8. Computation
case_run = CaseRunner(case, DOMAIN, comp_data_stat, all_domains={})
case_run.fill_computation_section()
```